<a href="https://colab.research.google.com/github/muhfirdaus67a/analisis-sentimen-cukai/blob/main/Pelabelan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title
import re
import csv
import pandas as pd
import os

class sentiStrength:
    def __init__(self):
        self.__sentiDict = self.load_dict("sentimenword.txt", tab=True)
        self.__emotDict = self.load_dict("emoticon.txt", tab=True)
        self.__negatingDict = self.load_list("negatingword.txt")
        self.__boosterDict = self.load_dict("boosterword.txt")
        self.__idiomDict = self.load_dict("idiomBaru.txt", tab=True)
        self.__questionDict = self.load_list("questionword.txt")
        self.unknown_words = []

    def load_list(self, file_path):
        try:
            with open(file_path, encoding='utf-8') as f:
                return [line.strip() for line in f.readlines()]
        except FileNotFoundError:
            print(f"❌ File tidak ditemukan:", file_path)
            return []

    def load_dict(self, file_path, tab=False, default_score=None):
        result = {}
        try:
            with open(file_path, encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    if tab or (" " in line and default_score is None):
                        parts = line.split("\t" if tab else " ")
                        if len(parts) == 2:
                            try:
                                result[parts[0]] = int(parts[1])
                            except ValueError:
                                pass
                    else:
                        if default_score is not None:
                            result[line] = default_score
        except FileNotFoundError:
            print(f"❌ File tidak ditemukan:", file_path)
        return result

    def main(self, text):
        tweet = text.split()

        pos, neg = 0, 0
        sentence_score = []
        isQuestion = False

        for i, term in enumerate(tweet):
            score = 0
            term_clean = term.lower()
            bigram = f"{tweet[i-1].lower()} {term_clean}" if i > 0 else ""

            # Cek emoticon / tanda baca / kata umum
            if term_clean in self.__emotDict:
                score = self.__emotDict[term_clean]

            elif term_clean in self.__sentiDict:
                # Kata ada di kamus utama
                score = self.__sentiDict[term_clean]

                # Negasi
                if i > 0 and tweet[i-1].lower() in self.__negatingDict:
                    score = -score

                # Booster sebelum
                if i > 0 and tweet[i-1].lower() in self.__boosterDict:
                    score += self.__boosterDict[tweet[i-1].lower()]

                # Booster sesudah
                elif i < len(tweet)-1 and tweet[i+1].lower() in self.__boosterDict:
                    score += self.__boosterDict[tweet[i+1].lower()]

                # Idiom / bigram
                if bigram in self.__idiomDict:
                    score = self.__idiomDict[bigram]

                # Kata tanya
                if term_clean in self.__questionDict:
                    isQuestion = True

            else:
                # bukan emoticon, bukan di kamus → unknown
                self.unknown_words.append(term_clean)

            sentence_score.append(f"{term_clean} [{score}]" if score != 0 else term_clean)

            if score > 0:
                pos += score
            elif score < 0:
                neg += score

        if abs(pos) > abs(neg):
            result_label = "Positif"
        elif abs(pos) < abs(neg) and not isQuestion:
            result_label = "Negatif"
        else:
            result_label = "Netral"

        return result_label, pos, neg, ' '.join(sentence_score)

    def save_unknown_words(self, filename="kata_tidak_dikenal.csv"):
        unique = sorted(set(self.unknown_words))
        with open(filename, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["kata_tidak_dikenal"])
            for w in unique:
                writer.writerow([w])
        print(f"[✓] {len(unique)} kata tidak dikenal disimpan ke '{filename}'")


def proses_csv_sentimen(file_csv, output_csv="hasil_sentimen.csv", unknown_csv="kata_tidak_dikenal.csv"):
    ss = sentiStrength()

    if not os.path.exists(file_csv):
        print("❌ File input tidak ditemukan:", file_csv)
        return

    df = pd.read_csv(file_csv, delimiter=';', quoting=csv.QUOTE_NONE, on_bad_lines='skip', engine='python')

    if 'stemming' not in df.columns:
        print("❌ Kolom 'stemming' tidak ditemukan di file CSV.")
        return

    hasil_label, skor_positif, skor_negatif, skor_net, rincian_kata = [], [], [], [], []

    for idx, row in df.iterrows():
        teks = str(row['stemming'])
        label, pos, neg, rincian = ss.main(teks)
        hasil_label.append(label)
        skor_positif.append(pos)
        skor_negatif.append(neg)
        skor_net.append(pos + neg)
        rincian_kata.append(rincian)

    df['label_sentimen'] = hasil_label
    df['skor_positif'] = skor_positif
    df['skor_negatif'] = skor_negatif
    df['skor_total'] = skor_net
    df['rincian_kata'] = rincian_kata

    df.to_csv(output_csv, sep=";", index=False, encoding='utf-8-sig')
    print(f"[✓] Hasil sentimen disimpan ke '{output_csv}'")
    ss.save_unknown_words(unknown_csv)


if __name__ == "__main__":
    proses_csv_sentimen("preprocessing.csv", "hasil_sentimen.csv", "kata_tidak_dikenal.csv")

❌ File tidak ditemukan: sentimenword.txt
❌ File tidak ditemukan: emoticon.txt
❌ File tidak ditemukan: negatingword.txt
❌ File tidak ditemukan: boosterword.txt
❌ File tidak ditemukan: idiomBaru.txt
❌ File tidak ditemukan: questionword.txt
❌ File input tidak ditemukan: preprocessing.csv


In [ ]:
import re
import csv
import pandas as pd
from collections import defaultdict
import os

# ======================= Spell Check Class =======================
class spellCheck:
    def train(self, features):
        model = defaultdict(lambda: 1)
        for f in features:
            model[f] += 1
        return model

    def __init__(self):
        try:
            with open('spellingset.txt', encoding='utf-8') as f:
                words = f.read()
        except FileNotFoundError:
            print("❌ File 'spellingset.txt' tidak ditemukan.")
            words = ""
        self.NWORDS = self.train(self.words(words))
        self.alphabet = 'abcdefghijklmnopqrstuvwxyz'

    def words(self, text):
        return re.findall('[a-z]+', text.lower())

    def edits1(self, word):
        splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
        deletes = [a + b[1:] for a, b in splits if b]
        transposes = [a + b[1] + b[0] + b[2:] for a, b in splits if len(b) > 1]
        replaces = [a + c + b[1:] for a, b in splits for c in self.alphabet if b]
        inserts = [a + c + b for a, b in splits for c in self.alphabet]
        return set(deletes + transposes + replaces + inserts)

    def known_edits2(self, word):
        return set(e2 for e1 in self.edits1(word) for e2 in self.edits1(e1) if e2 in self.NWORDS)

    def known(self, words):
        return set(w for w in words if w in self.NWORDS)

    def correct(self, word):
        candidates = self.known([word]) or self.known(self.edits1(word)) or self.known_edits2(word) or [word]
        return max(candidates, key=self.NWORDS.get)

# ======================= Normalisasi Kata Slang =======================
def load_slang_dict(file_path='slangword.txt'):
    slang_dict = {}
    if not os.path.exists(file_path):
        print(f"❌ File slang tidak ditemukan: {file_path}")
        return slang_dict
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(":")
            if len(parts) == 2:
                slang_dict[parts[0]] = parts[1]
    return slang_dict


def normalize_slang(text, slang_dict):
    return ' '.join([slang_dict.get(word, word) for word in text.split()])

# ======================= Sentiment Analyzer Class =======================
class sentiStrength:
    def __init__(self):
        self.__sentiDict = self.load_dict("sentimenword.txt", tab=True)
        self.__emotDict = self.load_dict("emoticon.txt", tab=True)
        self.__negatingDict = self.load_list("negatingword.txt")
        self.__boosterDict = self.load_dict("boosterword.txt")
        self.__idiomDict = self.load_dict("idiomBaru.txt", tab=True)
        self.__questionDict = self.load_list("questionword.txt")
        self.__katadasar = self.load_list("rootword.txt")
        self.__slangDict = load_slang_dict()
        self.spell_checker = spellCheck()
        self.unknown_words = []

    def load_list(self, file_path):
        try:
            with open(file_path, encoding='utf-8') as f:
                return [line.strip() for line in f.readlines()]
        except FileNotFoundError:
            print(f"❌ File tidak ditemukan: {file_path}")
            return []

    def load_dict(self, file_path, tab=False, default_score=None):
        result = {}
        try:
            with open(file_path, encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    if tab or (" " in line and default_score is None):
                        parts = line.split("\t" if tab else " ")
                        if len(parts) == 2:
                            try:
                                result[parts[0]] = int(parts[1])
                            except ValueError:
                                pass
                    else:
                        if default_score is not None:
                            result[line] = default_score
        except FileNotFoundError:
            print(f"❌ File tidak ditemukan: {file_path}")
        return result

    def main(self, text):
        tweet = text.split()
        tweet = [self.__slangDict.get(w, w) for w in tweet]

        pos, neg = 0, 0
        sentence_score = []
        isQuestion = False

        for i, term in enumerate(tweet):
            score = 0
            bigram = f"{tweet[i-1]} {term}" if i > 0 else ""
            term = re.sub(r'(\w+)-(\w+)', r'\1', term)

            if term not in self.__katadasar:
                corrected = self.spell_checker.correct(term)
                if corrected != term:
                    print(f"[KOREKSI] {term} → {corrected}")
                    term = corrected
                else:
                    self.unknown_words.append(term)

            if term.isalpha():
                if term in self.__sentiDict:
                    score = self.__sentiDict[term]
                    if i > 0 and tweet[i-1] in self.__negatingDict:
                        score = -score
                    if i > 0 and tweet[i-1] in self.__boosterDict:
                        score += self.__boosterDict[tweet[i-1]]
                    elif i < len(tweet)-1 and tweet[i+1] in self.__boosterDict:
                        score += self.__boosterDict[tweet[i+1]]
                if bigram in self.__idiomDict:
                    score = self.__idiomDict[bigram]
                if term in self.__questionDict:
                    isQuestion = True
            else:
                if term in self.__emotDict:
                    score = self.__emotDict[term]
                elif "?" in term:
                    isQuestion = True
                elif "!" in term:
                    score = 2
                elif re.sub(r'(\w+)[!]+$', r'\1', term) in self.__sentiDict:
                    score = self.__sentiDict[term]
                    score += 1 if score > 0 else -1

            sentence_score.append(f"{term} [{score}]" if score != 0 else term)

            if score > 0:
              pos += score
            elif score < 0:
              neg += score

        if abs(pos) > abs(neg):
            result_label = "Positif"
        elif abs(pos) < abs(neg) and not isQuestion:
            result_label = "Negatif"
        else:
            result_label = "Netral"

        return result_label, pos, neg, ' '.join(sentence_score)

    def save_unknown_words(self, filename="kata_tidak_dikenal.csv"):
        unique_words = list(set(self.unknown_words))
        with open(filename, mode='w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(["kata_tidak_dikenal"])
            for word in unique_words:
                writer.writerow([word])
        print(f"[✓] {len(unique_words)} kata tidak dikenal disimpan di '{filename}'")

# ======================= MODE 1: Proses CSV =======================
def proses_csv_sentimen(file_csv, output_csv="hasil_sentimen.csv", unknown_csv="kata_tidak_dikenal.csv"):
    slang_dict = load_slang_dict()
    ss = sentiStrength()

    if not os.path.exists(file_csv):
        print(f"❌ File input tidak ditemukan: {file_csv}")
        return

    df = pd.read_csv(file_csv, delimiter=';', quoting=csv.QUOTE_NONE, on_bad_lines='skip', engine='python')

    if 'stemming' not in df.columns:
        print("❌ Kolom 'stemming' tidak ditemukan di file CSV.")
        return

    hasil_label, skor_positif, skor_negatif, skor_net, skor_total, rincian_kata = [], [], [], [], [], []

    for idx, row in df.iterrows():
        teks = str(row['stemming'])
        teks = normalize_slang(teks, slang_dict)
        label, pos, neg, rincian = ss.main(teks)
        hasil_label.append(label)
        skor_positif.append(pos)
        skor_negatif.append(neg)
        skor_net.append(pos + neg)
        skor_total.append(pos + abs(neg))
        rincian_kata.append(rincian)

    df['label_sentimen'] = hasil_label
    df['skor_positif'] = skor_positif
    df['skor_negatif'] = skor_negatif
    df['skor_total'] = skor_net
    df['rincian_kata'] = rincian_kata

    df.to_csv(output_csv, sep=";", index=False, encoding='utf-8-sig')
    print(f"\n[✓] Hasil sentimen disimpan ke '{output_csv}'")
    ss.save_unknown_words(unknown_csv)

# ======================= MODE 2: Input Teks Manual =======================
def analisis_teks_input():
    ss = sentiStrength()
    slang_dict = load_slang_dict()

    while True:
        teks = input("\nMasukkan kalimat (atau ketik 'exit' untuk keluar): ").strip()
        if teks.lower() == "exit":
            break
        teks = normalize_slang(teks, slang_dict)
        label, pos, neg, rincian = ss.main(teks)
        skor_net = pos + neg
        skor_total = pos + abs(neg)
        print(f"\nHasil Analisis Sentimen:")
        print(f"- Label Sentimen : {label}")
        print(f"- Skor Positif   : {pos}")
        print(f"- Skor Negatif   : {neg}")
        print(f"- Skor Total     : {skor_net}")
        print(f"- Rincian Kata   : {rincian}")

    ss.save_unknown_words("kata_tidak_dikenal_input.csv")

# ======================= MAIN =======================
if __name__ == "__main__":
    # Langsung proses file CSV yang sudah diupload tanpa prompt interaktif.
    # Ubah nama file di bawah jika input/output berbeda.
    nama_file_input = "preprocessing.csv"
    nama_file_output = "hasil_sentimen.csv"
    nama_file_unknown = "kata_tidak_dikenal.csv"
    proses_csv_sentimen(nama_file_input, nama_file_output, nama_file_unknown)


[KOREKSI] hisap → bisa
[KOREKSI] paslon → pasien
[KOREKSI] minerral → mineral
[KOREKSI] apbn → apan
[KOREKSI] bpjs → baju
[KOREKSI] srm → sam
[KOREKSI] boba → coba
[KOREKSI] dst → ust
[KOREKSI] prohe → prove
[KOREKSI] vape → cape
[KOREKSI] le → ke
[KOREKSI] minerale → mineral
[KOREKSI] capres → cape
[KOREKSI] mandang → pandang
[KOREKSI] goceng → goreng
[KOREKSI] sachet → sahut
[KOREKSI] pulak → pula
[KOREKSI] golda → goda
[KOREKSI] esteh → entah
[KOREKSI] bpjs → baju
[KOREKSI] mulung → murung
[KOREKSI] vape → cape
[KOREKSI] marimas → maria
[KOREKSI] ale → alen
[KOREKSI] skm → smk
[KOREKSI] himbauan → hambatan
[KOREKSI] abai → abadi
[KOREKSI] standart → standar
[KOREKSI] kemenkeu → temenku
[KOREKSI] alfamart → alamat
[KOREKSI] luhut → lutut
[KOREKSI] tito → gito
[KOREKSI] eneg → eng
[KOREKSI] apbn → apan
[KOREKSI] iqbal → tebal
[KOREKSI] sales → bales
[KOREKSI] ikn → ika
[KOREKSI] nahh → nah
[KOREKSI] bubble → bule
[KOREKSI] press → dress
[KOREKSI] manggut → mangut
[KOREKSI] mulyani → m

In [ ]:
from google.colab import drive
drive.mount('/content/drive')